# 03 · Zadanie I3: AL z losowym wyborem kafli na CAMELYON17 (segmentacja jąder)

Odpowiednik notebooka 03 dla MoNuSeg. Student Cellpose-SAM (start: encoder SAM ViT-L, losowa głowica, czyli model B z notebooka 02)
uczy się na kaflach wybieranych **losowo**. „Adnotatorem” jest wyrocznia `cpsam`: student uczy się na jej maskach.

Zmiany względem MoNuSeg, bo **CAMELYON17 nie ma masek jąder**:
- jakość studenta to **zgodność z wyrocznią** (PQ względem jej masek) na kaflach val, a nie względem GT,
- poprzeczka nie jest „PQ wyroczni” (które wynosiłoby 1), tylko **samozgodność wyroczni** z notebooka 02 (PQ maski i maski z odbitego kafla),
  pomnożona przez `TARGET_FRAC`. Stop: PQ studenta na val ≥ poprzeczka albo koniec puli,
- jedyny pomiar względem prawdziwego GT to opcjonalny test transferu na MoNuSeg (na końcu).

Symulacja AL:
- pula: kafle train (zaadnotowane slajdy, podział po pacjencie), walidacja: kafle val, test nietknięty do końca,
- w każdej rundzie dochodzi `step` losowych kafli (zbiory zagnieżdżone, kolejność ustala seed),
- w każdej rundzie student trenowany **od nowa** z inicjalizacji SAM (cold start, tak samo dla wszystkich selektorów),
- student nigdy nie startuje z checkpointu Cellpose.

Kod: `al/camelyon17_random_loop.py`. Uruchamiaj na GPU. Wyniki lądują w `outputs/al_random_c17/<mode>/seed<k>/`;
ponowne uruchomienie pomija ukończone rundy.

In [ ]:
# --- działa bez instalacji pakietu: dodaje src/ repo do ścieżki importu ---
import sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "histopathology_pnw").exists())
sys.path.insert(0, str(_root / "src"))

## Konfiguracja

- `MODE = "smoke"`: 2 → 4 kafle, 1 epoka, kilka kafli val. Sprawdza tylko, że wszystko działa (minuty). PQ wyjdzie 0.
- `MODE = "full"`: pełny eksperyment. Budżet na rundę: 200 epok × 64 wycinki (jak w MoNuSeg), stały w każdej rundzie.
  Parametry nie były strojone na CAMELYON17. Jeśli PQ po pierwszych rundach stoi blisko 0, zwiększ `n_epochs` (tylko na podstawie val).
- `TARGET_FRAC`: jaką część samozgodności wyroczni student ma osiągnąć, żeby uznać cel za osiągnięty (domyślnie 0.95).

Pamięć GPU: jak w MoNuSeg, pętla włącza gradient checkpointing na blokach encodera (`gradient_checkpointing=True`).
`BATCH_SIZE = 4` mieści się w 16 GB. Przed uruchomieniem zamknij inne kernele z modelami na GPU (`nvidia-smi`).
Uruchamianie w tle: `AL_MODE=full nohup uv run --with nbconvert jupyter nbconvert --to notebook --execute --inplace
--ExecutePreprocessor.timeout=-1 notebooks/03_camelyon17_random_al.ipynb > outputs/al_random_c17.log 2>&1 &`.

In [ ]:
import os

# przed pierwszym użyciem CUDA: mniej fragmentacji pamięci GPU
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

MODE = os.environ.get("AL_MODE", "smoke")  # "smoke" albo "full"
SEEDS = [0, 1, 2]  # docelowo co najmniej 3 seedy, np. [0, 1, 2]
BATCH_SIZE = 4  # przy OOM zmniejsz do 2
TARGET_FRAC = 0.95
MIN_NUCLEI = 10  # kafle, na których wyrocznia znalazła mniej jąder, są odrzucane z puli (Cellpose pomija kafle < 5 masek)
print("MODE:", MODE)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cellpose import io as cp_io

from histopathology_pnw.al import camelyon17_random_loop as al
from histopathology_pnw.config import Settings
from histopathology_pnw.datasets import camelyon17 as c17
from histopathology_pnw.datasets import monuseg
from histopathology_pnw.device import get_device
from histopathology_pnw.inference import predict
from histopathology_pnw.metrics import evaluate_all, summarize
from histopathology_pnw.models.oracle import load_oracle
from histopathology_pnw.models.student import load_student

cp_io.logger_setup()  # logi treningu Cellpose (loss co epokę)
device = get_device()
print("device:", device)

if MODE == "smoke":
    configs = {s: al.TileALConfig(seed=s, initial=2, step=2, max_labeled=4, n_epochs=1, nimg_per_epoch=2,
                                  batch_size=1, stop_at_target=False) for s in SEEDS}
else:
    configs = {s: al.TileALConfig(seed=s, initial=4, step=4, max_labeled=None, n_epochs=200,
                                  nimg_per_epoch=64, batch_size=BATCH_SIZE) for s in SEEDS}
configs[SEEDS[0]]

In [ ]:
manifest = c17.extract_tiles(per_group=4)  # te same kafle co w notebooku 02
pool_all = c17.list_tiles("train")["tile_id"].tolist()
val = c17.list_tiles("val")["tile_id"].tolist()
test = c17.list_tiles("test")["tile_id"].tolist()
if MODE == "smoke":
    pool_all, val, test = pool_all[:6], val[:4], test[:4]
print(f"kafle: pula {len(pool_all)}, val {len(val)}, test {len(test)}")

## Adnotacje wyroczni i poprzeczka

Wyrocznia oznacza całą pulę (raz, cache w `outputs/oracle_masks/camelyon17/`). Z puli usuwamy kafle z < `MIN_NUCLEI` jąder.
Poprzeczka = `TARGET_FRAC` × samozgodność wyroczni na val (z notebooka 02, a jeśli jej nie ma, liczona od nowa).

In [ ]:
oracle = load_oracle(device)
all_ids = pool_all + val + test
oracle_masks = dict(zip(all_ids, al.oracle_tile_masks(oracle, all_ids), strict=True))

n_nuc = {t: len(np.unique(m)) - 1 for t, m in oracle_masks.items()}
pool = [t for t in pool_all if n_nuc[t] >= MIN_NUCLEI]
print(f"pula po filtrze: {len(pool)} / {len(pool_all)} kafli")
print("rundy (liczba oznaczonych kafli):", al.round_sizes(configs[SEEDS[0]], len(pool)))

sc_path = Settings().outputs_dir / "i1_c17" / f"oracle_self_consistency_{MODE}.json"
if sc_path.exists():
    SELFCONS_PQ = json.loads(sc_path.read_text())["val"]["pq"]
else:
    SELFCONS_PQ = al.oracle_self_consistency(oracle, val, [oracle_masks[t] for t in val])[0]["pq"]
TARGET_PQ = TARGET_FRAC * SELFCONS_PQ
print(f"samozgodność wyroczni (val PQ): {SELFCONS_PQ:.4f} -> poprzeczka dla studenta: {TARGET_PQ:.4f}")

## Pętla AL

Przerwany przebieg można wznowić, uruchamiając komórkę ponownie.

In [ ]:
results = {}
for seed, cfg in configs.items():
    out = Settings().outputs_dir / "al_random_c17" / MODE / f"seed{seed}"
    results[seed] = al.run_random_al(cfg, pool, val, oracle_masks, out, TARGET_PQ, device).assign(seed=seed)
results = pd.concat(results.values(), ignore_index=True)
results[["seed", "n_labeled", "val_pq", "val_sq", "val_dq", "val_aji", "final_train_loss", "train_time_s"]].round(4)

## Krzywa uczenia: zgodność studenta z wyrocznią vs liczba oznaczonych kafli

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for metric, a in zip(("val_pq", "val_aji"), ax, strict=True):
    for seed, g in results.groupby("seed"):
        a.plot(g["n_labeled"], g[metric], "o-", label=f"student, seed {seed}")
    if metric == "val_pq":
        a.axhline(SELFCONS_PQ, color="k", ls="--", label="samozgodność wyroczni")
        a.axhline(TARGET_PQ, color="gray", ls=":", label=f"poprzeczka ({TARGET_FRAC:.0%})")
    a.set_xlabel("liczba oznaczonych kafli")
    a.set_ylabel(metric.removeprefix("val_").upper() + " względem wyroczni (val)")
    a.legend()
plt.tight_layout()
fig.savefig(Settings().outputs_dir / "al_random_c17" / MODE / "learning_curve.png", dpi=120)

In [ ]:
reached = results[results["val_pq"] >= TARGET_PQ]
if len(reached):
    print("pierwsza runda, w której student osiągnął poprzeczkę:")
    print(reached.groupby("seed")["n_labeled"].min())
else:
    print(f"student nie osiągnął poprzeczki ({TARGET_PQ:.4f}); najlepiej: {results['val_pq'].max():.4f}")

## Końcowa ewaluacja (jeden raz)

Najlepsza runda wybrana **na val**. Potem:
1. zgodność z wyrocznią na kaflach **test** (obok samozgodności wyroczni na teście),
2. **opcjonalnie: transfer na MoNuSeg**, czyli ten sam student oceniony względem prawdziwych masek GT (14 obrazów testowych).
   To jedyna liczba w tym notebooku, którą można interpretować jako jakość segmentacji względem człowieka.
   Uwaga: Cellpose-SAM widział MoNuSeg, ale student nie (start z SAM), więc ten wynik nie jest zawyżony tą samą drogą co wynik wyroczni.

In [ ]:
from pathlib import Path

best = results.loc[results["val_pq"].idxmax()]
print(f"najlepsza runda: seed {best['seed']}, {best['n_labeled']} kafli, val PQ {best['val_pq']:.4f}")
student = load_student(Path(best["checkpoint"]), device)

test_imgs = [c17.read_tile(t) for t in test]
test_gt = [oracle_masks[t] for t in test]
student_test = summarize(evaluate_all(test_gt, predict(student, test_imgs), test))
oracle_test = al.oracle_self_consistency(oracle, test, test_gt)[0]
final = pd.DataFrame({"student vs wyrocznia": student_test, "samozgodność wyroczni": oracle_test}).T
final.to_csv(Settings().outputs_dir / "al_random_c17" / MODE / "final_test_c17.csv")
final.round(4)

In [ ]:
# transfer na MoNuSeg: student (uczony na maskach wyroczni z CAMELYON17) vs prawdziwe GT
mn = monuseg.list_split("test")
mn_pred = predict(student, [monuseg.read_image(s.image_path) for s in mn])
mn_gt = [monuseg.read_mask(s) for s in mn]
mn_table = evaluate_all(mn_gt, mn_pred, [s.image_id for s in mn])
transfer = summarize(mn_table)
pd.Series(transfer).round(4).to_frame("student (CAMELYON17) na MoNuSeg test, GT")

## Jak czytać wyniki

- Oś Y to **zgodność z wyrocznią**, nie dokładność. Krzywa mówi, jak szybko student „odtwarza” wyrocznię przy rosnącej
  liczbie kafli. Dla porównania selektorów AL (IT2 FLS, MC-Dropout, Coreset, VAAL) to dobra miara, bo wszystkie uczą się z tego samego „adnotatora”.
- Sufitem jest samozgodność wyroczni, nie 1. Student może ją przekroczyć tylko przypadkiem (szum).
- Test transferu na MoNuSeg odpowiada na pytanie, czy to, czego student nauczył się z masek wyroczni na CAMELYON17, ma
  jakąkolwiek wartość względem prawdziwych jąder. Porównaj go z wyrocznią na MoNuSeg (notebook 02 dla MoNuSeg), pamiętając o zastrzeżeniu o wycieku.
- Jeden seed to za mało: docelowo `SEEDS = [0, 1, 2]` (albo 5) i średnia ± odchylenie. Pula to kilkadziesiąt kafli z kilku slajdów,
  więc wynik mocno zależy od tego, które kafle (i z jakiego ośrodka) trafią do pierwszych rund.
- Kafle z jednego slajdu są skorelowane. Podział jest po pacjencie (val/test nie dzielą pacjentów z pulą), ale wewnątrz puli
  losowanie kafli ignoruje, z którego slajdu pochodzą.
- Hiperparametry są wspólne dla wszystkich rund i selektorów. Zmieniaj je tylko na podstawie val, nigdy testu.